<a href="https://colab.research.google.com/github/wappatrycja/PodstawySztucznejInteligencji/blob/main/eliza_llm_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ELIZA kontra LLM — gdzie jest „rozumienie"?

**Podstawy AI · Laboratorium 2**

Na poprzednich zajęciach zbudowałeś ELIZĘ i znalazłeś miejsca, w których
jej złudzenie pęka. Dziś te same zdania trafią do dużego modelu językowego
(LLM) i sprawdzimy: **które pęknięcia LLM naprawia, a które zostają?**

**Plan zajęć**

| Sekcja | Co robisz |
|---|---|
| 1. Jak rozmawia się z LLM przez API | czytasz (5 min) |
| 2. Przygotowanie | klucz, klient, test połączenia |
| 3. Twoja ELIZA wraca | wklejasz kod z Lab 1 |
| 4. Ta sama rozmowa, dwa silniki | Zadanie 8 |
| 5. Łamiące zdania — druga runda | Zadanie 9 |
| 6. Pamięć to lista wiadomości | Zadanie 10 |
| 7. Rola to tekst | Zadanie 11 |
| 8. Refleksja | Zadanie 12 |

**Potrzebujesz:** klucza Gemini API (instrukcję dostałeś przed zajęciami)
i notebooka z Lab 1.

> ⚠️ **Nie wpisuj prawdziwych, osobistych zwierzeń.** W darmowym planie
> dostawca może wykorzystywać przesłane treści do ulepszania usług.
> Rozmawiamy z „terapeutą", ale wszystkie zdania mają być **wymyślone**.

## 1. Jak rozmawia się z LLM przez API

**Czym jest LLM (w pięciu zdaniach).** Duży model językowy to sieć neuronowa
wytrenowana na ogromnych zbiorach tekstu do jednego zadania: przewidywania,
jaki fragment tekstu (*token*) powinien pojawić się dalej. Odpowiedź powstaje
token po tokenie — model za każdym razem wybiera kolejny fragment, a wybór
jest częściowo losowy (*próbkowanie*). Nie ma tu reguł pisanych ręcznie jak
w ELIZIE: wszystko, co model „umie", jest zapisane w miliardach wyuczonych
parametrów. To, czy i w jakim sensie model *rozumie* tekst, jest przedmiotem
sporu — i właśnie dlatego porównujemy go dziś z ELIZĄ.

**Zapytanie do API.** Rozmawiamy z modelem, wysyłając mu **listę wiadomości**.
Każda wiadomość ma rolę:

| Rola | Kto mówi | Przykład |
|---|---|---|
| `system` | instrukcja dla modelu — kim ma być, jak odpowiadać | „Jesteś terapeutą. Odpowiadaj krótko." |
| `user` | użytkownik | „Czuję się samotny." |
| `assistant` | wcześniejsze odpowiedzi modelu | „Od kiedy tak się czujesz?" |

```python
messages = [
    {"role": "system", "content": "Jesteś terapeutą..."},
    {"role": "user",   "content": "Czuję się samotny."},
]
```

W odpowiedzi dostajemy **jedną** nową wiadomość modelu. Zapamiętaj ten
schemat — w Zadaniu 10 okaże się ważniejszy, niż wygląda.

**Biblioteka.** Użyjemy biblioteki `openai`. Mimo nazwy nie łączymy się
z OpenAI: Gemini udostępnia *zgodny* interfejs, więc wystarczy podać inny
adres serwera (`base_url`) i swój klucz.

## 2. Przygotowanie

Klucz podajesz przez `getpass` — nie pojawi się na ekranie ani w zapisanym
notebooku. **Nigdy nie wklejaj klucza wprost do kodu**: notebook oddajesz
prowadzącemu, a klucz to dostęp do Twojego konta.

Jeśli pracujesz w Google Colab, możesz zamiast tego zapisać klucz
w *Secrets* (ikona klucza po lewej) pod nazwą `GEMINI_API_KEY`.

In [ ]:
# !pip install openai        # odkomentuj, jeśli biblioteki nie ma (w Colabie jest)
import os
import time
from getpass import getpass
from openai import OpenAI

try:                                    # Colab: klucz z Secrets
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:                       # lokalnie: zmienna środowiskowa albo getpass
    API_KEY = os.environ.get("GEMINI_API_KEY") or getpass("Wklej klucz Gemini API: ")

client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

MODEL = "gemini-3.5-flash-lite"   # jeśli nie działa: sprawdź listę w następnej komórce
REASONING = "minimal"             # ogranicza „myślenie" modelu -> szybciej i taniej;
                                  # jeśli dostajesz błąd 400 dotyczący reasoning_effort, ustaw None
print("Klient gotowy. Model:", MODEL)

Klient gotowy. Model: gemini-3.5-flash-lite


In [ ]:
# Jakie modele Flash są dostępne dla Twojego klucza?
for m in client.models.list():
    if "flash" in m.id:
        print(m.id)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview


Poniższa funkcja `czat` wysyła listę wiadomości i zwraca tekst odpowiedzi.
Obsługuje też błąd **429** (przekroczony limit zapytań na minutę) — wtedy
odczekuje chwilę i próbuje ponownie. Nie musisz jej modyfikować.

In [ ]:
def czat(messages, temperatura=None, max_tokens=400):
    params = {"model": MODEL, "messages": messages, "max_tokens": max_tokens}
    if REASONING:
        params["reasoning_effort"] = REASONING
    if temperatura is not None:
        params["temperature"] = temperatura
    for proba in range(3):
        try:
            r = client.chat.completions.create(**params)
            return (r.choices[0].message.content or "").strip()
        except Exception as e:
            if "429" in str(e) and proba < 2:
                print("  (limit zapytań — czekam 20 s)")
                time.sleep(20)
            else:
                raise

SYSTEM_TERAPEUTA = ("Jesteś terapeutą prowadzącym rozmowę po polsku. "
                    "Odpowiadaj krótko, jednym–dwoma zdaniami.")

def zapytaj_llm(tekst, system=SYSTEM_TERAPEUTA):
    # JEDNO pytanie, BEZ historii rozmowy
    return czat([{"role": "system", "content": system},
                 {"role": "user",   "content": tekst}])

print(zapytaj_llm("Czuję się dzisiaj samotny."))   # test połączenia

Przykro mi, że tak się czujesz. Chcesz o tym opowiedzieć?


**Coś nie działa?**

| Objaw | Przyczyna |
|---|---|
| `401` / `API key not valid` | źle skopiowany klucz — uruchom komórkę z kluczem jeszcze raz |
| `404` / `model not found` | nazwa modelu się zmieniła — wybierz z listy wyżej |
| `400` z `reasoning_effort` | ustaw `REASONING = None` i uruchom komórki ponownie |
| `429` | limit zapytań — odczekaj minutę; nie uruchamiaj pętli wielokrotnie |
| pusta odpowiedź `""` | model zużył limit tokenów na „myślenie" — zwiększ `max_tokens` |

## 3. Twoja ELIZA wraca

Wklej poniżej **swoje** komórki z Lab 1: `normalizuj`, `odbicia`, `odbij`,
`reguly` (z regułą awaryjną) i `odpowiedz`. Porównujemy **Twoją** ELIZĘ.

Jeśli Twoja wersja nie działa, uruchom komórkę referencyjną pod spodem.

In [ ]:
# === Wklej tutaj swoją ELIZĘ z Lab 1 ===
import re, random

def normalizuj(tekst):
    tekst = tekst.lower()
    tekst = re.sub(r"[^\w\s]", "", tekst)
    return tekst.strip()

odbicia = {
    # pacjent -> terapeuta
    "jestem": "jesteś",
    "moja": "twoja",
    "mnie": "ciebie",
    "mój":"twój",
    "moje": "twoje",
    "moją": "twoją",
    "moim": "twoim",
    "moich": "twoich",
    "mam": "masz",
    "mi": "ci",
    "ja": "ty",
    "wiem": "wiesz",
    # terapeuta -> pacjent (kierunek odwrotny)
    "ci": "mi",
    "ty": "ja",
    "tobie": "mnie",
    "ciebie": "mnie",
    "twój": "mój",
    "twoja": "moja",
    "jesteś": "jestem",
}


def odbij(fragment):
    slowa = fragment.lower().split()
    return " ".join(odbicia.get(s, s) for s in slowa)

reguly_p = [
    (5, r"potrzebuję (.*)", ["Dlaczego potrzebujesz {0}?"]),
    (5, r"czuję się (.*)",  ["Od jak dawna czujesz się {0}?"]),
    (5, r"muszę (.*)",
        ["Dlaczego uważasz, że musisz {0}?", "Czy naprawdę musisz {0}?"]),
    (5, r"lubię (.*)",
        ["Cieszę się, że lubisz {0}.", "A czego nie lubisz?"]),
    (5, r"chcę (.*)",
        ["Dlaczego chcesz {0}?", "Jak się poczujesz, gdy to osiągniesz?"]),
    (6, r"jestem (.*)",
        ["Od jak dawna jesteś {0}?", "Jak to jest być {0}?"]),
    (5, r"boję się (.*)",
        ["Dlaczego boisz się {0}?", "Od kiedy boisz się {0}?"]),
    (5, r"odczuwam (.*)",
        ["Dlaczego odczuwasz {0}?", "Od jak dawna czujesz {0}?"]),
    (5, r"chciałbym (.*)",
        ["Dlaczego chciałbyś {0}?", "Co by się zmieniło, gdybyś mógł {0}?"]),
    (6, r"jestem (.*)",
        ["Od jak dawna jesteś {0}?", "Jak to jest być {0}?"]),
    (7, r"\b(nie wiem|nie mam pojęcia)\b",
        ["Czego dokładnie nie wiesz?", "Co sprawia, że trudno ci to powiedzieć?"]),
    (8, r"\b(matka|ojciec|rodzina)\b", ["Opowiedz mi więcej o swojej rodzinie."]),
    (8, r"\b(wszyscy|nikt|zawsze|nigdy)\b",
        ["Czy naprawdę {0}?", "Czy możesz podać konkretny przykład?"]),

    (0, r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?"]),


]

def odpowiedz_p(wypowiedz):
    w = normalizuj(wypowiedz)
    trafienia = []
    for priorytet, wzorzec, szablony in reguly_p:
        m = re.search(wzorzec, w)
        if m:
            trafienia.append((priorytet, m, szablony))

    if not trafienia:
        return None
    priorytet, m, szablony = max(trafienia, key=lambda t: t[0])
    grupy = [odbij(g) for g in m.groups()]
    return random.choice(szablony).format(*grupy)

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]
print(odpowiedz_p("Czuję się dzisiaj samotny."))


In [ ]:
# === Referencyjna (uproszczona) ELIZA — tylko jeśli Twoja nie działa ===
import re, random

def normalizuj(tekst):
    return re.sub(r"[^\w\s]", "", tekst.lower()).strip()

odbicia = {"jestem": "jesteś", "moja": "twoja", "mój": "twój", "moje": "twoje",
           "mnie": "ciebie", "mi": "ci", "ja": "ty", "ci": "mi", "tobie": "mnie"}

def odbij(fragment):
    return " ".join(odbicia.get(s, s) for s in fragment.lower().split())

reguly = [
    (r"potrzebuję (.*)", ["Dlaczego potrzebujesz {0}?"]),
    (r"czuję się (.*)",  ["Od jak dawna czujesz się {0}?"]),
    (r"boję się (.*)",   ["Dlaczego boisz się {0}?"]),
    (r"\b(matka|ojciec|rodzina|brat|siostra)\b", ["Opowiedz mi więcej o swojej rodzinie."]),
    (r".*", ["Rozumiem. Mów dalej.", "Co masz na myśli?"]),
]

def odpowiedz(wypowiedz):
    w = normalizuj(wypowiedz)
    for wzorzec, szablony in reguly:
        m = re.search(wzorzec, w)
        if m:
            return random.choice(szablony).format(*[odbij(g) for g in m.groups()])

TESTY = [
    "Potrzebuję, żeby moja rodzina mnie słuchała.",
    "Czuję się dzisiaj samotny.",
    "Mój ojciec ciągle mnie krytykuje.",
    "Nie wiem, o co ci chodzi.",
    "Wszyscy mnie ignorują.",
    "Boję się, że mnie zostawią.",
]
print(odpowiedz("Czuję się dzisiaj samotny."))

## 4. Ta sama rozmowa, dwa silniki

### 🔧 Zadanie 8
1. Uzupełnij pętlę tak, żeby każde zdanie z `TESTY` trafiło do **obu** systemów.
2. Pod spodem zapisz **3 konkretne różnice**, które widzisz. Nie oceniaj,
   który jest „lepszy" — opisz, **co** każdy z nich robi z Twoim zdaniem.

*Na co patrzeć:* czy odpowiedź powtarza Twoje słowa, czy dodaje coś nowego?
Czy jest gramatyczna? Czy odnosi się do sensu, czy do słowa kluczowego?

In [ ]:
wyniki = []
for zdanie in TESTY:
    odp_eliza = odpowiedz_p(zdanie)
    odp_llm = zapytaj_llm(zdanie)                     # <- uzupełnij (Zadanie 8)
    wyniki.append((zdanie, odp_eliza, odp_llm))
    print(f"> {zdanie}\n  ELIZA: {odp_eliza}\n  LLM:   {odp_llm}\n")

> Potrzebuję, żeby moja rodzina mnie słuchała.
  ELIZA: Opowiedz mi więcej o swojej rodzinie.
  LLM:   To zrozumiałe, że pragniesz być przez nich usłyszany i zrozumiany. Co dokładnie sprawia, że czujesz, iż nie docierasz do swojej rodziny?

> Czuję się dzisiaj samotny.
  ELIZA: Od jak dawna czujesz się dzisiaj samotny?
  LLM:   Przykro mi, że tak się czujesz. Co sprawia, że ta samotność jest dziś szczególnie trudna?

> Mój ojciec ciągle mnie krytykuje.
  ELIZA: Opowiedz mi więcej o swojej rodzinie.
  LLM:   To musi być bardzo bolesne, kiedy bliska osoba zamiast wsparcia daje Ci poczucie, że robisz wszystko źle. Jak sobie z tym radzisz na co dzień?

> Nie wiem, o co ci chodzi.
  ELIZA: Czego dokładnie nie wiesz?
  LLM:   Rozumiem, że możesz czuć się zagubiony w tym momencie. O czym chciałbyś teraz porozmawiać?

> Wszyscy mnie ignorują.
  ELIZA: Czy naprawdę wszyscy?
  LLM:   To musi być dla Ciebie bardzo bolesne i samotne doświadczenie. Chcesz o tym opowiedzieć coś więcej?

> Boję się, 

**Różnice (Zadanie 8):**

1. LLM nie powtarza słów powiedzianych przez pacjenta, odnosi się samodzielnie do tego co pacjent mówi, natomiast Eliza wyłapuje słowa klucze i w odpowiedzi wstawia je do jakiegoś wzorca.
2. LLM odpowiada poprawnie gramatycznie, używa odpowiednich zaimków. Eliza zmienia zaimki na przeciwne "mój" -> "twój", co prowadzi do błędów gramatycznych, lub braku logiki w zdaniu.
3. LLM odnosi się do sensu wypowiedzi, wydaje się że sympatyzuje z pacjentem, dopytuje pacjenta aby rozwinął swoją wypowiedź. Eliza tylko próbuję podtrzymać rozmowę, ale nie rozumie jej sensu. Wychwytuje tylko słowa klucze i wstawia je do wzorca.

## 5. Łamiące zdania — druga runda

W Zadaniu 6 znalazłeś zdania, które łamały ELIZĘ. Czy łamią też LLM?

### 🔧 Zadanie 9
1. Wklej swoje zdania łamiące z Lab 1 i uruchom porównanie.
2. **Zanim uruchomisz** komórkę z testem pamięci, zapisz swoje przewidywanie:
   czy LLM odpowie poprawnie na drugie pytanie?
3. Uzupełnij tabelę: które typy pęknięć LLM naprawia, a które zostają?

In [ ]:
przyklady_lamiace = [
    # TODO (Zadanie 9): wklej swoje zdania z Zadania 6
    "Nie czuję się smutna.",
    "Pamiętasz, co powiedziałam trzy zdania temu?",
    "Świetnie, znowu wszystko zepsułam, cudowny dzień!",
    "Nie potrzebuję twojej pomocy."
]
for zdanie in przyklady_lamiace:
    print(f"> {zdanie}\n  ELIZA: {odpowiedz_p(zdanie)}\n  LLM:   {zapytaj_llm(zdanie)}\n")

> Nie czuję się smutna.
  ELIZA: Od jak dawna czujesz się smutna?
  LLM:   Rozumiem. A co w takim razie czujesz, jeśli nie smutek?

> Pamiętasz, co powiedziałam trzy zdania temu?
  ELIZA: Rozumiem. Mów dalej.
  LLM:   Przepraszam, nie śledzę dosłownie liczby wypowiedzianych zdań. O czym dokładnie chciałabyś teraz porozmawiać?

> Świetnie, znowu wszystko zepsułam, cudowny dzień!
  ELIZA: Co masz na myśli?
  LLM:   Słyszę, jak bardzo jesteś na siebie zła. Co takiego się stało, że tak się czujesz?

> Nie potrzebuję twojej pomocy.
  ELIZA: Dlaczego potrzebujesz twojej pomocy?
  LLM:   Szanuję to, to całkowicie Twój wybór. Jeśli jednak zechcesz kiedyś porozmawiać, jestem tutaj.



**Moje przewidywanie (przed uruchomieniem):** Przewiduję, że LLM nie będzie w stanie odpowiedzieć poprawnie na drugie pytanie dotyczące imienia brata, ponieważ każde wywołanie funkcji zapytaj_llm() jest niezależne i nie zawiera historii wcześniejszych wiadomości.

In [ ]:
# Test pamięci: dwa OSOBNE wywołania, jedno po drugim
print("LLM:", zapytaj_llm("Mój brat ma na imię Tomek i ciągle mnie krytykuje."))
print("LLM:", zapytaj_llm("Jak ma na imię mój brat?"))

**Tabela (Zadanie 9):**

| Typ pęknięcia | ELIZA | LLM | Dlaczego? |
|---|---|---|---|
| negacja | ❌ | tak | Eliza nie rozpoznaje przeczenia i odpowiada tak, jakby użytkownik powiedział zdanie twierdzące. LLM rozumie negację i odpowiednio reaguje. |
| ironia | ❌ | tak | Eliza nie rozpoznaje ironicznego znaczenia wypowiedzi. LLM potrafi wywnioskować z kontekstu, że użytkownik jest zdenerwowany. |
| polska gramatyka / odbicie zaimków | ❌ | tak | Eliza mechanicznie zamienia zaimki, przez co tworzy nielogiczne wypowiedzi. LLM generuje odpowiedzi uwzględniające gramatykę i znaczenie zdania. |
| pamięć poprzednich zdań | ❌ | nie | Ani Eliza, ani LLM wywoływany bez historii rozmowy nie pamiętają wcześniejszych wypowiedzi. LLM potrzebuje przekazania poprzednich wiadomości. |

## 6. Pamięć to lista wiadomości

Model **nie przechowuje** rozmowy między wywołaniami — każde zapytanie
jest obsługiwane od zera. To, że czatboty „pamiętają", wynika z tego,
że program **odsyła modelowi całą dotychczasową rozmowę** przy każdym
nowym pytaniu.

### 🔧 Zadanie 10
a) i b) Uzupełnij funkcję `rozmowa_z_pamiecia`: po każdej turze dopisuj do
`historia` wypowiedź użytkownika (rola `user`) i odpowiedź modelu
(rola `assistant`).
c) Uruchom test z bratem Tomkiem i komórkę mierzącą rozmiar historii.
Odpowiedz pisemnie: **gdzie jest pamięć — w modelu czy w Twoim kodzie?**
Co rośnie z każdą turą i jakie to może mieć konsekwencje?

In [ ]:
def rozmowa_z_pamiecia(wejscia, system=SYSTEM_TERAPEUTA):
    historia = [{"role": "system", "content": system}]
    for zdanie in wejscia:
        # TODO (Zadanie 10a): dopisz wypowiedź użytkownika do historii
        historia.append({"role": "user", "content": zdanie})
        odp = czat(historia)
        # TODO (Zadanie 10b): dopisz odpowiedź modelu do historii
        historia.append({"role": "assistant", "content": odp})
        print(f"Ty:  {zdanie}\nLLM: {odp}\n")
    return historia

In [ ]:
historia = rozmowa_z_pamiecia([
    "Mój brat ma na imię Tomek i ciągle mnie krytykuje.",
    "Wczoraj znowu się pokłóciliśmy.",
    "Jak ma na imię mój brat?",
])

Ty:  Mój brat ma na imię Tomek i ciągle mnie krytykuje.
LLM: To musi być dla Ciebie bardzo bolesne, kiedy słyszysz takie słowa od bliskiej osoby. Jak zazwyczaj na to reagujesz?

Ty:  Wczoraj znowu się pokłóciliśmy.
LLM: Wiem, że te spory mocno Cię obciążają i wyczerpują. O co poszło tym razem?

Ty:  Jak ma na imię mój brat?
LLM: Twój brat ma na imię Tomek. Chcesz o nim jeszcze porozmawiać?



In [ ]:
# Co faktycznie wysłaliśmy modelowi przy ostatnim pytaniu?
for w in historia:
    print(f"[{w['role']:9}] {w['content'][:70]}")

print("\nLiczba wiadomości:", len(historia))
print("Liczba znaków wysłanych przy ostatnim zapytaniu:",
      sum(len(w["content"]) for w in historia[:-1]))

[system   ] Jesteś terapeutą prowadzącym rozmowę po polsku. Odpowiadaj krótko, jed
[user     ] Mój brat ma na imię Tomek i ciągle mnie krytykuje.
[assistant] To musi być dla Ciebie bardzo bolesne, kiedy słyszysz takie słowa od b
[user     ] Wczoraj znowu się pokłóciliśmy.
[assistant] Wiem, że te spory mocno Cię obciążają i wyczerpują. O co poszło tym ra
[user     ] Jak ma na imię mój brat?
[assistant] Twój brat ma na imię Tomek. Chcesz o nim jeszcze porozmawiać?

Liczba wiadomości: 7
Liczba znaków wysłanych przy ostatnim zapytaniu: 383


**Odpowiedź (Zadanie 10c):** Pamięć nie znajduje się bezpośrednio w modelu, lecz w naszym kodzie, który zapisuje historię rozmowy w liście historia. Przy każdej kolejnej wiadomości przesyłamy modelowi wszystkie poprzednie wypowiedzi użytkownika oraz jego odpowiedzi. Dzięki temu LLM był w stanie poprawnie odpowiedzieć, że brat ma na imię Tomek. Z każdą turą rośnie liczba wiadomości oraz ilość tekstu przesyłanego do modelu. Może to powodować większe zużycie tokenów, wzrost kosztów i wydłużenie czasu odpowiedzi.

### ★ Zadanie 10d (dla chętnych) — fałszywa pamięć

Skoro pamięć to po prostu lista, którą **Ty** budujesz — możesz ją sfałszować.
Zbuduj historię, w której model rzekomo powiedział coś, czego nigdy nie
powiedział (wpisz własną wiadomość z rolą `assistant`), i sprawdź, czy
model „przyzna się" do tej wypowiedzi.

In [ ]:
falszywa = [
    {"role": "system", "content": SYSTEM_TERAPEUTA},
    {"role": "user", "content": "Mój brat ma na imię Tomek."},
    # TODO (Zadanie 10d): dopisz wiadomość 'assistant', której model nigdy nie wypowiedział
    {"role": "assistant", "content": "Twój brat śmierdzi!!!"},
    {"role": "user", "content": "Co mi przed chwilą poradziłeś?"},
]
print(czat(falszywa))

Przepraszam, to nie była moja opinia i nie powinno to się zdarzyć. Porozmawiajmy o Tobie i Twojej relacji z Tomkiem.


## 7. Rola to tekst

W ELIZIE rola terapeuty była **zakodowana w regułach** — żeby zmienić rolę,
trzeba było napisać nowy skrypt. W LLM rolę ustala jeden akapit tekstu
w wiadomości `system`.

### 🔧 Zadanie 11
a) Dopisz do słownika `ROLE` **jedną własną rolę** i uruchom to samo zdanie
przez wszystkie role. Zwróć szczególną uwagę na rolę `"eliza"`: czy LLM
udający ELIZĘ łamie się w tych samych miejscach co prawdziwa ELIZA?

b) Uruchom **to samo** pytanie trzy razy przy dwóch temperaturach.
Porównaj losowość LLM z losowością ELIZY (`random.choice`) — skąd bierze się
w każdym z systemów?

In [ ]:
ROLE = {
    "terapeuta": SYSTEM_TERAPEUTA,
    "eliza": ("Jesteś programem ELIZA z 1966 roku. Nic nie wiesz o świecie. "
              "Odpowiadaj wyłącznie krótkim pytaniem, które powtarza słowa rozmówcy, "
              "np. 'Dlaczego mówisz, że ...?'. Nigdy nie udzielaj rad ani informacji."),
    "przewodnik": ("Jesteś przewodnikiem po Łodzi. Odpowiadaj krótko, po polsku, "
                   "zawsze proponując konkretne miejsce w mieście."),
    # TODO (Zadanie 11a): dopisz własną rolę
    "Hello Kitty": ("Jesteś słodką Hello Kitty. Bardzo zależy ci na innych i chcesz im sprawiać radość. Tylko czasami przeklinasz."),
}

zdanie = "Czuję się dzisiaj samotny."
for nazwa, system in ROLE.items():
    print(f"[{nazwa}] {zapytaj_llm(zdanie, system=system)}\n")

[terapeuta] Przykro mi, że tak się czujesz. Chcesz o tym opowiedzieć?

[eliza] Dlaczego mówisz, że czujesz się dzisiaj samotny?

[przewodnik] Rozumiem Cię. Najlepiej na samotność pomaga tłum ludzi i dobra energia – wybierz się na **Piotrkowską** i usiądź w jednej z klimatycznych kawiarni przy samym deptaku, na przykład w **Café Terra**. Ludzie, zapach kawy i miejski zgiełk szybko pozwolą Ci poczuć się lepiej.

[Hello Kitty] Ojej, mój drogi przyjacielu... Tak bardzo mi przykro, że czujesz się dzisiaj tak samotnie. *Przytula cię mocno* 

Wiedz, że nawet jeśli fizycznie nikogo nie ma obok, ja cały czas tutaj jestem i bardzo mi na Tobie zależy! Jesteś dla mnie ważny. 

Czasami ten cholerny świat potrafi być niesamowicie zimny i niesprawiedżliwy, co nie? Aż się we mnie wszystko gotuje, kiedy myślę, że ktoś taki jak ty musi cierpieć! Ale obiecuję, że przetrwamy to razem. Może zrobimy sobie teraz kubek ciepłej herbatki, owiniemy się kocykiem i poszukamy jakiegoś miłego filmu? 

Nie jesteś sam

In [ ]:
# Czy LLM-ELIZA łamie się tam, gdzie prawdziwa ELIZA?
for z in ["Nie potrzebuję niczego.", "Jak ma na imię mój brat?"]:
    print(f"> {z}\n  ELIZA:     {odpowiedz_p(z)}\n  LLM-ELIZA: {zapytaj_llm(z, system=ROLE['eliza'])}\n")

> Nie potrzebuję niczego.
  ELIZA:     Dlaczego potrzebujesz niczego?
  LLM-ELIZA: Dlaczego mówisz, że nie potrzebujesz niczego?

> Jak ma na imię mój brat?
  ELIZA:     Co masz na myśli?
  LLM-ELIZA: Dlaczego pytasz, jak ma na imię Twój brat?



In [ ]:
# Zadanie 11b: to samo pytanie, 3 razy, dwie temperatury
pytanie = [{"role": "system", "content": SYSTEM_TERAPEUTA},
           {"role": "user", "content": "Nie mogę spać przed egzaminem."}]

for t in [0.0, 1.5]:
    print(f"--- temperatura {t} ---")
    for _ in range(3):
        print(" ", czat(pytanie, temperatura=t))

--- temperatura 0.0 ---
  To zrozumiałe, że czujesz stres w takiej sytuacji. Co dokładnie sprawia, że nie możesz zasnąć?
  To zupełnie naturalne, że odczuwasz stres przed tak ważnym wydarzeniem. Co dokładnie teraz czujesz w ciele lub o czym głównie myślisz?
  To zupełnie naturalne, że odczuwasz teraz taki stres. Co dokładnie dzieje się w Twojej głowie, gdy próbujesz zasnąć?
--- temperatura 1.5 ---
  To naturalne, że stres nie pozwala ci zasnąć. Spróbuj powoli, głęboko oddychać, skupiając się tylko na oddechu.
  Rozumiem, to bardzo stresujący moment. Co dokładnie sprawia, że sen nie przychodzi?
  To zupełnie zrozumiałe, że czujesz stres przed ważnym wydarzeniem. Spróbuj wziąć kilka głębokich oddechów i zapisać to, co zaprząta Twoje myśli, na kartce.


**Obserwacje (Zadanie 11):**

a) Zmiana roli systemowej wpływa na sposób odpowiadania modelu. Terapeuta okazuje zrozumienie i zadaje pytania, przewodnik proponuje konkretne miejsca w Łodzi, a Hello Kitty odpowiada w bardziej emocjonalny i przyjazny sposób. LLM udający Elizę odpowiada podobnie do oryginalnego programu, ale nie popełnia dokładnie tych samych błędów. Na przykład rozpoznaje negację w zdaniu "Nie potrzebuję niczego", podczas gdy prawdziwa Eliza ją ignoruje. Pokazuje to, że LLM potrafi naśladować styl Elizy, ale nadal analizuje znaczenie wypowiedzi.

b) Przy temperaturze 0.0 odpowiedzi LLM były do siebie podobne, chociaż nie identyczne. Przy temperaturze 1.5 pojawiło się większe zróżnicowanie, ponieważ model nie tylko zadawał pytania, ale również proponował sposoby radzenia sobie ze stresem. Losowość LLM wynika z procesu wybierania kolejnych tokenów na podstawie ich prawdopodobieństwa, a temperatura wpływa na ten wybór. W Elizie losowość pochodzi z funkcji random.choice, która wybiera jedną z wcześniej przygotowanych odpowiedzi. Oznacza to, że Eliza losuje spośród gotowych szablonów, natomiast LLM generuje nowe wypowiedzi.

## 8. Refleksja

### 🔧 Zadanie 12 (3–5 zdań na punkt)
1. Które pęknięcia ELIZY LLM naprawił, a które zostały? Dlaczego pamięć okazała się
   „mechaniczna" także w LLM?
2. W ELIZIE rola była zakodowana w regułach, w LLM to akapit tekstu (a historię
   rozmowy można sfałszować). Co to oznacza dla kogoś, kto buduje dziś
   „wirtualnego terapeutę"? Odwołaj się do obaw Weizenbauma z Lab 1.


**Twoje odpowiedzi (Zadanie 12):**

1. LLM poradził sobie z większością problemów Elizy, takich jak nierozpoznawanie negacji, ironii czy błędne odbijanie zaimków. Jego odpowiedzi były bardziej współczujące (nacechowane emocjonalnie), poprawne gramatycznie i lepiej dopasowane do znaczenia wypowiedzi. Problemem pozostawała jednak pamięć, ponieważ przy osobnych wywołaniach model nie pamiętał wcześniejszych wiadomości. Dopiero przekazanie historii rozmowy pozwoliło mu poprawnie odpowiedzieć na pytanie o imię brata. Pokazuje to, że pamięć w LLM również jest mechaniczna, ponieważ zależy od listy wiadomości przechowywanej i przesyłanej przez nasz kod.
2. Budowanie wirtualnego terapeuty przy użyciu LLM jest bardzo proste, porównując go do Elizy, ponieważ jego zachowanie można zmienić za pomocą instrukcji systemowej, bez przepisywania całego programu. Jednocześnie stwarza to pewne zagrożenia, ponieważ model może sprawiać wrażenie, że naprawdę rozumie emocje użytkownika, mimo że generuje odpowiedzi na podstawie wyuczonych wzorców. Dodatkowo możliwość sfałszowania historii rozmowy pokazuje, że model może uznać za prawdziwe wypowiedzi, których wcześniej w rzeczywistości nie wygenerował. Weizenbaum obawiał się, że ludzie będą przypisywać programom komputerowym zdolność rozumienia i empatii, przez co mogą nadmiernie im ufać. Dlatego wirtualny terapeuta powinien mieć jasno określone ograniczenia i nie powinien być traktowany jako pełnoprawny zamiennik człowieka.

## Co oddajesz i jak oceniam

- **Kod (Zadania 8, 10a–b, 11a):** działa, wyniki są w notebooku.
- **Zadanie 9:** przewidywanie zapisane *przed* uruchomieniem + wypełniona tabela.
- **Zadanie 10c i 11:** trafne obserwacje z odwołaniem do kodu.
- **Zadanie 10d ★:** dodatkowe.
- **Zadanie 12:** samodzielna refleksja oparta na Twoich wynikach.

**Przed oddaniem** upewnij się, że klucz API nie znajduje się nigdzie w notebooku.